## Interactive: PCA and dialogism side by side

Explore both scoring methods together for any book, zoomed to any line range: the standardized signal plot (`viz.plot_overlay`, with speech regions shaded and labeled) and, optionally, the underlying text with speech-like features highlighted (`viz.highlighted_excerpt`, driven by `dialogism`'s ranked lexicons).

Training PCA happens once, with fixed sample size and seed — training variability itself is explored in `1 - Author and speech signal` and `Interactive PCA`, not here. Window size is the one thing worth revisiting a few times per session, since it changes both rolling scores; work/line-range selection and the highlighted-text controls are meant for fast, frequent back-and-forth.

### Import statements

In [ ]:
import ccc2026
from ccc2026 import dialogism, viz
ccc2026.setup()

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from matplotlib import pyplot as plt

### Train once

Fixed sample size and seed, full feature set for both classes. No widget here on purpose — see the intro above.

In [ ]:
PCA_SAMPLE_SIZE = 1000
PCA_SEED = 1

feature_set = {
    "lemma": ccc2026.top_lemmas,
    "pos": ccc2026.all_pos,
    "morph": ccc2026.top_morph,
}
train = ccc2026.run_training(feature_set, sample_size=PCA_SAMPLE_SIZE, seed=PCA_SEED)

lexicons = dialogism.build_lexicons()
dialogism_score = dialogism.token_dialogism_score(lexicons)

print("Trained.")

### Shared state and redraw functions

Defined before the widgets that use them, but referencing widgets defined further down — fine as long as the whole notebook is run top to bottom before anything is clicked, same pattern `Interactive PCA` already uses.

In [ ]:
state = {}

def cutoff_for_top_n(lexicon, n):
    '''Cutoff value such that (approximately) the top n ranked features clear
    it via a ">" comparison, as build_display_column expects.'''
    ranked = lexicon.sort_values(ascending=False)
    if n >= len(ranked):
        return ranked.min() - 1
    return ranked.iloc[n - 1] - 1e-9

def current_line_range():
    '''(first_line, last_line), treating the input boxes' untouched defaults
    (1 and 99999) as "no restriction" rather than a literal range — so the
    default whole-book view doesn't show a spurious "(1-99999)" in the title.
    '''
    first_line = first_line_input.value if first_line_input.value > 1 else None
    last_line = last_line_input.value if last_line_input.value < 99999 else None
    return first_line, last_line

def redraw_plot():
    with plot_output:
        clear_output(wait=True)
        if "pca_roll" not in state:
            display(HTML('<i>Set a window size and click "Apply window" first.</i>'))
            return
        first_line, last_line = current_line_range()
        fig = viz.plot_overlay(
            ccc2026.tokens, work_dd.value, pref_dd.value,
            state["pca_roll"], state["dialogism_roll"],
            first_line=first_line, last_line=last_line,
        )
        display(fig)
        plt.close(fig)

def redraw_text():
    with text_output:
        clear_output(wait=True)
        if not text_toggle.value:
            return
        if "pca_roll" not in state:
            display(HTML('<i>Set a window size and click "Apply window" first.</i>'))
            return
        cutoffs = (lemma_n_input.value, grammar_n_input.value)
        if state.get("display_cutoffs") != cutoffs:
            lemma_cutoff = cutoff_for_top_n(lexicons["lemma"], lemma_n_input.value)
            grammar_cutoff = cutoff_for_top_n(lexicons["grammar"], grammar_n_input.value)
            ccc2026.tokens["display"] = viz.build_display_column(
                ccc2026.tokens, lexicons,
                lemma_cutoff=lemma_cutoff, grammar_cutoff=grammar_cutoff,
            )
            state["display_cutoffs"] = cutoffs
        first_line, last_line = current_line_range()
        html = viz.highlighted_excerpt(
            ccc2026.tokens, work_dd.value, pref_dd.value,
            first_line=first_line, last_line=last_line,
        )
        display(HTML(html))

### Window size

The one control worth an explicit "apply" step rather than live updates — it rerolls both scores across the whole corpus.

In [ ]:
window_input = widgets.BoundedIntText(
    value=200, min=10, max=2000, step=10,
    description="Window size",
)
apply_window_btn = widgets.Button(description="Apply window")
window_status = widgets.Output()

def on_apply_window(btn):
    with window_status:
        clear_output(wait=True)
        print(f"Computing rolling scores at window={window_input.value}...")
    state["window"] = window_input.value
    state["pca_roll"] = ccc2026.rolling_samples(train, window_size=window_input.value)["speech_score"]["score"]
    state["dialogism_roll"] = dialogism.rolling_dialogism(dialogism_score, window_size=window_input.value)["speech_score"]["score"]
    with window_status:
        clear_output(wait=True)
        print(f"Ready: window={window_input.value}")
    redraw_plot()
    redraw_text()

apply_window_btn.on_click(on_apply_window)

### Select a passage

Work, book, and an optional line range — all live-updating, since re-slicing and replotting already-computed scores is cheap.

In [ ]:
work_dd = widgets.Dropdown(
    description="Work",
    options=list(ccc2026.all_prefs),
)
pref_dd = widgets.Dropdown(
    description="Book",
    options=ccc2026.all_prefs[work_dd.value],
)
first_line_input = widgets.IntText(value=1, description="First line")
last_line_input = widgets.IntText(value=99999, description="Last line")
plot_output = widgets.Output()

def on_work_change(change):
    '''changing work updates the book options, which will itself trigger
    on_passage_change via pref_dd\'s own observer'''
    pref_dd.options = ccc2026.all_prefs[work_dd.value]

def on_passage_change(change):
    redraw_plot()
    redraw_text()

work_dd.observe(on_work_change, names="value")
pref_dd.observe(on_passage_change, names="value")
first_line_input.observe(on_passage_change, names="value")
last_line_input.observe(on_passage_change, names="value")

### Full text

Off by default — meant to be switched on once you've zoomed into a passage of a few dozen lines, not left on while browsing whole books.

In [ ]:
text_toggle = widgets.Checkbox(value=False, description="Show full text")
lemma_n_input = widgets.BoundedIntText(value=30, min=1, max=200, description="Top N lemmas")
grammar_n_input = widgets.BoundedIntText(value=10, min=1, max=50, description="Top N grammar")
text_output = widgets.Output()

def on_text_settings_change(change):
    redraw_text()

text_toggle.observe(on_text_settings_change, names="value")
lemma_n_input.observe(on_text_settings_change, names="value")
grammar_n_input.observe(on_text_settings_change, names="value")

## Interactive UI

In [ ]:
display(widgets.HBox([window_input, apply_window_btn]))
display(window_status)

In [ ]:
display(widgets.HBox([work_dd, pref_dd, first_line_input, last_line_input]))
display(plot_output)

In [ ]:
display(widgets.HBox([text_toggle, lemma_n_input, grammar_n_input]))
display(text_output)

In [ ]:
# populate everything on first run, rather than requiring a manual click
on_apply_window(None)